In [57]:
from pydantic import BaseModel
from typing import Literal
import random, json
from enum import Enum

class ReadQuest(BaseModel):
    quest_type: Literal["symbol","color"]
    quest_level: Literal[1,2,3]
    quest_words: list[str]

In [3]:
quest = []
quest.extend([
    ReadQuest(**{"quest_type":"symbol","quest_level":1,"quest_words":["고양이","강아지"]}),
    ReadQuest(**{"quest_type":"symbol","quest_level":3,"quest_words":["토끼"]}),
    ReadQuest(**{"quest_type":"symbol","quest_level":2,"quest_words":["개구리","백곰"]}),
    ReadQuest(**{"quest_type":"symbol","quest_level":3,"quest_words":["코끼리","펭귄"]}),
    ReadQuest(**{"quest_type":"color","quest_level":1,"quest_words":["노랑","파랑"]}),
    ReadQuest(**{"quest_type":"color","quest_level":2,"quest_words":["녹색"]}),
    ReadQuest(**{"quest_type":"color","quest_level":3,"quest_words":["빨강"]}),
])

In [43]:
def quest_words(quests:list[ReadQuest],_type:str,level:int):
    words = []
    for word in [q.quest_words for q in quests if q.quest_type == _type and q.quest_level == level]:
        words.extend(word)
    return words

In [5]:
symbols = quest_words("symbol",3)
words = quest_words("color",3)
symbols, words

(['토끼', '코끼리', '펭귄'], ['빨강'])

In [6]:
[(symbol,word) for symbol in symbols for word in words]

[('토끼', '빨강'), ('코끼리', '빨강'), ('펭귄', '빨강')]

In [ ]:
json_object = [q.dict() for q in quest]
print(json.dumps(json_object,indent=2,ensure_ascii=False))

[
  {
    "quest_type": "symbol",
    "quest_level": 1,
    "quest_words": [
      "고양이",
      "강아지"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 3,
    "quest_words": [
      "토끼"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 2,
    "quest_words": [
      "개구리",
      "백곰"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 3,
    "quest_words": [
      "코끼리",
      "펭귄"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 1,
    "quest_words": [
      "노랑",
      "파랑"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 2,
    "quest_words": [
      "녹색"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 3,
    "quest_words": [
      "빨강"
    ]
  }
]


/tmp/ipykernel_37593/908257399.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json_object = [q.dict() for q in quest]


In [84]:
quest_json = [
    {'quest_type': 'symbol', 'quest_level': 1, 'quest_words': ['고양이', '강아지', '사자', '호랑이']}, 
    {'quest_type': 'symbol', 'quest_level': 2, 'quest_words': ['개구리', '백곰']},
    {'quest_type': 'symbol', 'quest_level': 3, 'quest_words': ['코끼리', '펭귄', '토끼']},
    {'quest_type': 'color', 'quest_level': 1, 'quest_words': ['노랑','파랑','빨강']},
    {'quest_type': 'color', 'quest_level': 2, 'quest_words': ['녹색','남색']},
    {'quest_type': 'color', 'quest_level': 3, 'quest_words': ['주황']}
]
quest_json
print(json.dumps(quest_json,indent=2,ensure_ascii=False))

[
  {
    "quest_type": "symbol",
    "quest_level": 1,
    "quest_words": [
      "고양이",
      "강아지",
      "사자",
      "호랑이"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 2,
    "quest_words": [
      "개구리",
      "백곰"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 3,
    "quest_words": [
      "코끼리",
      "펭귄",
      "토끼"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 1,
    "quest_words": [
      "노랑",
      "파랑",
      "빨강"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 2,
    "quest_words": [
      "녹색",
      "남색"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 3,
    "quest_words": [
      "주황"
    ]
  }
]


In [47]:
quests = [ReadQuest(**quest) for quest in quest_json]

In [72]:
from common.ko_util import korean_to_english_pronunciation
quest_template = {
    'word_data1':"{} 스티커를 찾아라",
    'word_data2':"{} 캐리어를 찾아라",
    'full_data':"{} 스티커가 붙은 {} 캐리어를 찾아라"
}
class QuestLevel(Enum):
    EASY = 1
    NORMAL = 2
    HARD = 3
class WordData(BaseModel):
    kor: str
    eng: str
    pronunciation: str
class ReadTargetData(BaseModel):
    symbol:str
    color:str
class QuestInfo(BaseModel):
    index:int
    dificulity:QuestLevel
class QuestReadInfo(QuestInfo):
    target_data: list[ReadTargetData]
    correct_answer_index: int
    word_data1: WordData
    word_data2: WordData
    full_data: WordData

In [80]:
def gen_read_quest(quests:list[ReadQuest],quest_count:int = 10):
    symbols = quest_words(quests,'symbol',1)
    colors = quest_words(quests,'color',1)
    quest_data = random.sample([(symbol, color) for symbol in symbols for color in colors],quest_count)
    correct_index = random.randint(0,quest_count-1)
    target_data = [ReadTargetData(symbol=q_data[0],color=q_data[1]) for q_data in quest_data]
    word_data1 = quest_template['word_data1'].format(quest_data[correct_index][0])
    word_data2 = quest_template['word_data2'].format(quest_data[correct_index][1])
    full_data = quest_template['full_data'].format(*quest_data[correct_index])
    return QuestReadInfo(
        index=1,
        dificulity=QuestLevel.EASY,
        target_data=target_data,
        correct_answer_index=correct_index,
        word_data1=WordData(
            kor = word_data1,
            eng = '',
            pronunciation=korean_to_english_pronunciation(word_data1)
        ),
        word_data2=WordData(
            kor = word_data2,
            eng = '',
            pronunciation=korean_to_english_pronunciation(word_data2)
        ),
        full_data=WordData(
            kor = full_data,
            eng = '',
            pronunciation=korean_to_english_pronunciation(full_data)
        )
    )

In [82]:
read_scenario = gen_read_quest(quests)

In [83]:
read_scenario.word_data1

WordData(kor='강아지 스티커를 찾아라', eng='', pronunciation='gang-a-ji seu-ti-keo-reul chat-a-ra')

In [107]:
import ollama
def ko_to_en(ko:str):
    system_prompt = """
        당신은 영어 번역가 입니다.
        한글 문장을 영문으로 번역하여 해당 영문만 알려주세요.
    """
    user_prompt = "한글 문장을 영문으로 번역해줘 : {}"
    response = ollama.chat(
        model="hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M",
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt.format(ko)}
        ]
    )
    return response['message']['content']

In [114]:
ko_to_en('빨강색 캐리어를 찾아라')

'Please find a red car.'

In [1]:
skip_api_url = [
    ':8104/logs'
]
skip_body_url = [
    ':8104/files', ':8104/speaking', ':8104/write'
]

In [6]:
_url = 'http://100.100.53.32:8104/writes/questions'
any([skip_url in _url for skip_url in skip_body_url])

True